## Perceptron -- a simple example

In [ ]:

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Sigmoid (logistic) activation
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Perceptron with sigmoid
def perceptron(x, w, b):
    z = np.dot(x, w) + b
    return sigmoid(z)

# Example
x = np.array([2, 3])
w = np.array([1, -1])
b = 0.5

output = perceptron(x, w, b)
print("Sigmoid output:", output)

Sigmoid output: 0.3775406687981454


# Two-layer perceptron

$h=σ({\bf{w_1​^Tx}} + b_1​)$

$y=σ(w_2​h+b_2​)$

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
from PIL import Image
from IPython.display import display

img = Image.open("/content/drive/My Drive/courses_@IITD/ML4Chem_CML7211/perceptron_221.png")
resized_img = img.resize((500, 400))

display(resized_img)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# two perceptron, 2 --> 2 --> 1
def two_perceptron(x, w1, b1, w2, b2):
    h = sigmoid(np.dot(x, w1) + b1)

    # ouput
    y = sigmoid(np.dot(h, w2) + b2)

    return y

# input
x = np.array([2.0, 3.0])

# weights and biases
w1 = np.array([[1.0, -1.0],   # input --> hidden
               [-1.0,  1.0]])
b1 = np.array([0.0, 0.0])

w2 = np.array([1.0, -1.0])    # hidden --> output
b2 = 0.0

# Forward pass
output = two_perceptron(x, w1, b1, w2, b2)
print("Output:", output)

# Multi-Layer Perceptron (MLP)

In [ ]:
import matplotlib.pyplot as plt

# 2-2-1 architecture

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# dataset XOR problem
X = np.array([[0,0],
              [0,1],
              [1,0],
              [1,1]])

y = np.array([[0],[1],[1],[0]])

# Plot
plt.figure(figsize=(5,5))

for i in range(len(X)):
    if y[i] == 0:
        plt.scatter(X[i,0], X[i,1], marker='o', label='Class 0' if i==0 else "")
    else:
        plt.scatter(X[i,0], X[i,1], marker='x', label='Class 1' if i==1 else "")

plt.xlabel("x1")
plt.ylabel("x2")
plt.title("XOR Dataset")
plt.legend()
plt.grid(True)

plt.show()



In [ ]:

np.random.seed(42)

W1 = np.random.randn(2,2) * 0.1  # 2 × 2 matrix, values from a standard normal distribution: 𝑁(0,1)
W2 = np.random.randn(2,1) * 0.1  # 2 × 1 matrix

b1 = np.zeros((1, 2))
b2 = np.zeros((1, 1))

# Training
lr = 0.5
epochs = 50000

losses = []

for epoch in range(epochs):

    # forward
    z1 = np.dot(X, W1) + b1
    h  = sigmoid(z1)

    z2 = np.dot(h, W2) + b2
    y_pred = sigmoid(z2)

    # MSE Loss
    loss = np.mean((y - y_pred)**2)
    losses.append(loss)


    # back-propagation
    dL_dy = (y_pred - y)

    dL_dz2 = dL_dy * sigmoid_derivative(z2)
    dL_dW2 = np.dot(h.T, dL_dz2)
    dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

    dL_dh  = np.dot(dL_dz2, W2.T)
    dL_dz1 = dL_dh * sigmoid_derivative(z1)
    dL_dW1 = np.dot(X.T, dL_dz1)
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    # update weights
    W2 -= lr * dL_dW2
    b2 -= lr * dL_db2

    W1 -= lr * dL_dW1
    b1 -= lr * dL_db1

    if epoch % 1000 == 0:
       print(f"Epoch {epoch} | Loss = {loss:.4f} | dW2 = {dL_dW2.flatten()} | dW1 = {dL_dW1.flatten()}")
       print("\nFinal predictions:")
       print(y_pred)

plt.figure(figsize=(6,4))
plt.plot(losses)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.grid()
plt.show()


## IRIS data set example

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

df = iris.frame

print(df.head())      # first 5 rows
print(iris.keys())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split


X, y = load_iris(return_X_y=True)
y = y.reshape(-1,1)

# One-hot encoding
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)   # (n_samples, 3)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


def sigmoid(z):
    #z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def softmax(z):
    exp = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)


np.random.seed(42)

input_dim = X_train.shape[1]
hidden_dim = 5
output_dim = y_train.shape[1]

W1 = np.random.randn(input_dim, hidden_dim) * 0.05
b1 = np.zeros((1, hidden_dim))

W2 = np.random.randn(hidden_dim, output_dim) * 0.05
b2 = np.zeros((1, output_dim))

# Training
lr = 0.001
epochs = 100000

train_losses = []
test_losses = []

for epoch in range(epochs):

    # forward
    z1 = np.dot(X_train, W1) + b1
    h  = sigmoid(z1)

    z2 = np.dot(h, W2) + b2
    y_pred = softmax(z2)

    # test loss (no grad update)
    z1_test = np.dot(X_test, W1) + b1
    h_test  = sigmoid(z1_test)

    z2_test = np.dot(h_test, W2) + b2
    y_test_pred = softmax(z2_test)


    # cross-entropy train loss
    train_loss = -np.mean(np.sum(y_train * np.log(y_pred + 1e-8), axis=1))
    train_losses.append(loss)

    # test loss
    test_loss = -np.mean(np.sum(y_test * np.log(y_test_pred + 1e-8), axis=1))
    test_losses.append(test_loss)


    # backprop
    m = X_train.shape[0]

    dL_dz2 = (y_pred - y_train) / m

    dL_dW2 = np.dot(h.T, dL_dz2)
    dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

    dL_dh  = np.dot(dL_dz2, W2.T)
    dL_dz1 = dL_dh * sigmoid_derivative(z1)

    dL_dW1 = np.dot(X_train.T, dL_dz1)
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    # update
    W2 -= lr * dL_dW2
    b2 -= lr * dL_db2

    W1 -= lr * dL_dW1
    b1 -= lr * dL_db1

    if epoch % 1000 == 0:
        print(f"Epoch {epoch} | Train Loss = {loss:.4f} | Test Loss = {test_loss:.4f}")

# evaluation
z1 = np.dot(X_test, W1) + b1
h  = sigmoid(z1)
z2 = np.dot(h, W2) + b2
y_pred = softmax(z2)

pred_labels = np.argmax(y_pred, axis=1)
true_labels = np.argmax(y_test, axis=1)

accuracy = np.mean(pred_labels == true_labels)
print("\nTest Accuracy:", accuracy)

plt.figure(figsize=(6,4))
plt.plot(losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Train vs Test Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Forward pass on test data
z1 = np.dot(X_test, W1) + b1
h  = sigmoid(z1)
z2 = np.dot(h, W2) + b2
y_pred = softmax(z2)

# Convert to class labels
pred_labels = np.argmax(y_pred, axis=1)
true_labels = np.argmax(y_test, axis=1)

plt.figure(figsize=(12,5))

# True table
plt.subplot(1,2,1)
plt.scatter(X_test[:,0], X_test[:,1], c=true_labels)
plt.title("True Labels")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

# predicted
plt.subplot(1,2,2)
plt.scatter(X_test[:,0], X_test[:,1], c=pred_labels)
plt.title("Predicted Labels")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()

print("\nEnter a new sample (4 features): 5.1, 3.5, 1.4, 0.2")
print("Format: sepal_length sepal_width petal_length petal_width")


user_input = input("Input: ")
values = list(map(float, user_input.strip().split()))
new_sample = np.array([values])   # shape (1,4)

new_sample_scaled = scaler.transform(new_sample)

z1 = np.dot(new_sample_scaled, W1) + b1
h  = sigmoid(z1)

z2 = np.dot(h, W2) + b2
probs = softmax(z2)

pred_class = np.argmax(probs, axis=1)[0]
print("Predicted class:", pred_class)

from sklearn.datasets import load_iris

iris = load_iris()
print("Predicted species:", iris.target_names[pred_class])
print("Class probabilities:", probs)